# 09 — Construction des artefacts de production

**Objectif :** réunir, signer et recharger tout ce qui est nécessaire à l'inférence locale.

**Entrées :** artefacts des notebooks 04 à 07.  
**Sorties :** bundle local `artifacts/manifest.json` et rapport publiable sans poids.  
**Dépendance :** notebook 08.  
**Temps estimé :** 1 à 3 minutes.  
**Ressources :** CUDA pendant le smoke test d'inférence.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Racine du projet introuvable.")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.artifact_loader import build_manifest, validate_manifest
from src.recommender import HybridRecommender

PROCESSED_DIR = ROOT / "data" / "processed"
ARTIFACTS_DIR = ROOT / "artifacts"
REPORTS_DIR = ROOT / "reports"

## Catalogue et popularité

In [ ]:
products = pd.read_csv(PROCESSED_DIR / "products_clean.csv", dtype={"sku": str}).fillna("")
fit_frame = pd.read_csv(PROCESSED_DIR / "fit_clicks.csv", dtype={"sku": str}).fillna("")
products.to_csv(ARTIFACTS_DIR / "products.csv", index=False)

counts = fit_frame["sku"].value_counts()
maximum = max(float(counts.max()), 1.0)
popularity = {str(sku): float(count / maximum) for sku, count in counts.items()}
(ARTIFACTS_DIR / "popularity.json").write_text(
    json.dumps(popularity, ensure_ascii=False), encoding="utf-8"
)

## Manifeste d'intégrité

In [ ]:
filenames = [
    "products.csv",
    "click_history.json",
    "popularity.json",
    "word_tfidf.joblib",
    "char_tfidf.joblib",
    "word_product_matrix.joblib",
    "char_product_matrix.joblib",
    "semantic.index",
    "semantic_skus.json",
    "hybrid_config.json",
]
source = json.loads((REPORTS_DIR / "data_inventory.json").read_text(encoding="utf-8"))["source"]
manifest = build_manifest(
    ARTIFACTS_DIR,
    filenames,
    {
        "source": source,
        "redistribution_allowed": False,
        "note": "Vérifier la licence Kaggle avant toute publication des artefacts dérivés.",
    },
)
(ARTIFACTS_DIR / "manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
)
assert validate_manifest(ARTIFACTS_DIR) == []

## Rechargement et smoke test

In [ ]:
recommender = HybridRecommender(ARTIFACTS_DIR)
recommendations = recommender.recommend("space shooter", k=5)
assert len(recommendations) == 5
assert len({row["sku"] for row in recommendations}) == 5
display(pd.DataFrame(recommendations))

summary = {
    "source": source,
    "artifact_count": len(manifest["files"]),
    "total_bytes": sum(file["bytes"] for file in manifest["files"].values()),
    "integrity": "ok",
    "smoke_query": "space shooter",
    "smoke_skus": [row["sku"] for row in recommendations],
    "redistributable": False,
}
(REPORTS_DIR / "artifact_manifest_summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(json.dumps(summary, indent=2, ensure_ascii=False))

## Conclusion

Le bundle est local, vérifié par SHA-256 et exclu de Git. Sa redistribution reste désactivée tant
que la licence des données et des dérivés n'a pas été confirmée.